# Type a function signature

A checklist-style walkthrough for annotating a function: parameters (including defaults, `*args`, `**kwargs`), return type, and the small calls you might not have thought about (generators, async, context managers).

## The basic shape

For a plain function: annotate each parameter after `:`, the return after `->`. Use `None` for functions that return nothing meaningful.

In [ ]:
def save_to_file(data: str, path: str) -> None:
    with open(path, "w") as f:
        f.write(data)

save_to_file("hello", "/tmp/test.txt")

## Optional parameters with defaults

The default goes after the type. The type describes what values are valid — the default is just one example.

In [ ]:
def paginate(items: list, page: int = 1, per_page: int = 20) -> list:
    start = (page - 1) * per_page
    return items[start:start + per_page]

print(paginate([1, 2, 3, 4, 5, 6, 7, 8], page=2, per_page=3))

If the parameter is optional *and* can be `None`, the annotation needs `| None`:

In [ ]:
def fetch(url: str, timeout: float | None = None) -> str:
    actual_timeout = timeout if timeout is not None else 30.0
    return f"GET {url} (timeout={actual_timeout})"

print(fetch("http://example.com"))
print(fetch("http://example.com", timeout=5.0))

Common mistake: writing `timeout: float = None` — that's annotated as `float` but `None` doesn't match. `float | None = None` is correct.

## Variable and keyword arguments

`*args` and `**kwargs` annotate the element type, not the collection type. Inside the function `args` is a `tuple[T, ...]` and `kwargs` is a `dict[str, T]`.

In [ ]:
def format_row(*values: str, separator: str = ",") -> str:
    return separator.join(values)

print(format_row("a", "b", "c"))
print(format_row("x", "y", separator=" | "))

In [ ]:
def build_query(**params: str | int) -> str:
    # Each value is str or int
    return "&".join(f"{k}={v}" for k, v in params.items())

print(build_query(user_id=42, sort="name", limit=10))

## Keyword-only and positional-only

PEP 570 markers (`*` and `/`) in the signature don't affect the annotations — annotate each parameter normally:

In [ ]:
def connect(host: str, /, port: int = 80, *, timeout: float = 30) -> str:
    # 'host' is positional-only, 'timeout' is keyword-only
    return f"{host}:{port} (timeout={timeout})"

print(connect("example.com"))
print(connect("example.com", 443, timeout=5))

## Generator functions

A function with `yield` returns a `Generator[YieldType, SendType, ReturnType]` — or, if you only care about what it yields, `Iterator[YieldType]`:

In [ ]:
from collections.abc import Iterator

def countdown(n: int) -> Iterator[int]:
    while n > 0:
        yield n
        n -= 1

for i in countdown(3):
    print(i)

Use `Iterator[T]` for the common case. `Generator[Y, S, R]` matters only if the generator consumes values via `.send()` or has a meaningful `return`. See the [iterators-and-generators guide](../../iterators-and-generators/) for when each applies.

## Async functions

Annotate the *awaited* value, not a coroutine wrapper. The compiler handles the wrapping:

In [ ]:
import asyncio

async def fetch_data(url: str) -> dict:
    # Pretend network call
    await asyncio.sleep(0)
    return {"url": url, "status": 200}

result = asyncio.run(fetch_data("http://example.com"))
print(result)

`-> dict` means "awaiting this gives a dict". The return type of the function object is `Coroutine[Any, Any, dict]`, but you virtually never need to write that out — the type-checker understands `async def`.

## Overloads — when the return type depends on the input

Sometimes a single function returns different types depending on its arguments. `@overload` lets you declare multiple signatures for the same function:

In [ ]:
from typing import overload

@overload
def parse(value: str) -> int: ...
@overload
def parse(value: str, *, as_float: bool) -> float: ...

def parse(value: str, *, as_float: bool = False) -> int | float:
    # The real implementation — only the overloads above are seen by callers
    return float(value) if as_float else int(value)

x = parse("42")             # type-checker sees: int
y = parse("3.14", as_float=True)  # type-checker sees: float
print(x, type(x))
print(y, type(y))

The `@overload` stubs are declarations only — the implementation at the bottom is what actually runs. The type-checker picks the matching overload based on the call, and reports the appropriate return type to callers.

## Complete example

Putting it together — a real-ish function with everything in one place:

In [ ]:
from collections.abc import Callable, Iterable
from typing import TypeVar

T = TypeVar("T")
R = TypeVar("R")

def batch_apply(
    fn: Callable[[T], R],
    items: Iterable[T],
    *,
    batch_size: int = 100,
    on_error: Callable[[T, Exception], None] | None = None,
) -> list[R]:
    """Apply `fn` to each item, collecting results.
    Call `on_error(item, exc)` on failure if provided, else propagate.
    """
    results: list[R] = []
    for item in items:
        try:
            results.append(fn(item))
        except Exception as e:
            if on_error is None:
                raise
            on_error(item, e)
    return results

# Every argument and return is precisely typed:
#  - fn: a callable taking one T and returning an R
#  - items: an iterable of Ts
#  - batch_size: int, keyword-only with a default
#  - on_error: optional callback, or None
#  - return: a list of Rs

print(batch_apply(lambda s: len(s), ["hello", "world", "!"]))
print(batch_apply(str.upper, ["abc", "def"]))

## Tips

- **Annotate the return, always.** The return is the contract with callers; a missing annotation makes the function `Any` at the call site.
- **Public functions first.** If you're gradually annotating a codebase, start with the ones that get called from many places.
- **Prefer abstract parameter types, concrete return types.** `Iterable[T]` in, `list[T]` out — flexibility for callers, clarity for downstream users.
- **Don't over-specify.** `list[int]` when the function only iterates is too restrictive — use `Iterable[int]`.